In [1]:
!pip install av torchvision pytorchvideo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.7/132.7 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 44.0 MB/s eta 0:00:00
  Created wheel for pytorchvideo: filename=pytorchvideo-0.1.5-py3-none-any.whl size=188686 sha256=c1b41929db2543a1d8b6a37b0abe965510091064baa30033083b4337b8a8e876
  Stored in directory: /root/.cache/pip/wheels/b3/49/dc/aab2dce83e38b59849db13a4f4ddd220e568e24b58332fb0f9
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=77dcd306fbb70742c650f430db10f85fcb99cc2b1479ddc4c46aa55a34b9438d
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel f

In [1]:
# --- 1. IMPORTS ---
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.io import read_video
import torchvision.transforms as T
from google.colab import drive

# --- 2. CONFIGURATION ---
drive.mount('/content/drive', force_remount=True)

DATA_PATH = "/content/drive/MyDrive/video_data/train"
CLASSES = ['walking', 'running']
BATCH_SIZE = 2
# SlowFast typically prefers 32 frames for the fast pathway
NUM_FRAMES = 32
EPOCHS = 3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 3. DATASET DEFINITION ---
class VideoDataset(Dataset):
    def __init__(self, root, classes, transform=None, num_frames=32):
        self.samples = []
        self.transform = transform
        self.num_frames = num_frames

        for i, cls_name in enumerate(classes):
            cls_path = os.path.join(root, cls_name)
            if not os.path.isdir(cls_path): continue
            for f in os.listdir(cls_path):
                if f.lower().endswith(('.mp4', '.avi', '.mov')):
                    self.samples.append((os.path.join(cls_path, f), i))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            # vframes shape: (T, C, H, W)
            vframes, _, _ = read_video(path, pts_unit='sec', output_format='TCHW')
            t = vframes.shape[0]
            if t == 0: raise ValueError("Empty Video")

            # Uniform sampling of 32 frames
            indices = torch.linspace(0, t - 1, self.num_frames).long()
            vframes = vframes[indices]

            # Apply transforms to (T, C, H, W)
            if self.transform:
                vframes = self.transform(vframes)

            # Permute to standard video tensor: (C, T, H, W)
            vframes = vframes.permute(1, 0, 2, 3)

            return vframes, label
        except Exception as e:
            print(f"Error loading {path}: {e}")
            return torch.zeros((3, self.num_frames, 256, 256)), label

# --- 4. MODEL & TRANSFORMS INITIALIZATION ---
# Standard normalization for PyTorchVideo models
slowfast_transform = T.Compose([
    T.Resize((256, 256)),
    T.ConvertImageDtype(torch.float32),
    T.Normalize(mean=[0.45, 0.45, 0.45], std=[0.225, 0.225, 0.225])
])

train_dataset = VideoDataset(DATA_PATH, CLASSES, transform=slowfast_transform, num_frames=NUM_FRAMES)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Load the SlowFast-R50 model from PyTorch Hub
print("Downloading SlowFast model from PyTorch Hub...")
model = torch.hub.load('facebookresearch/pytorchvideo', 'slowfast_r50', pretrained=True)

# Replace the classification head. In SlowFast, this is block 6.
in_features = model.blocks[6].proj.in_features
model.blocks[6].proj = nn.Linear(in_features, len(CLASSES))

model = model.to(DEVICE)

# --- 5. TRAINING LOOP ---
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

print(f"\n--- Starting Training ({len(train_dataset)} videos) ---")
model.train()

# Alpha is the frame rate ratio between the Fast and Slow pathways.
# If Fast is 32 frames, Slow takes every 4th frame (8 frames).
ALPHA = 4

for epoch in range(EPOCHS):
    running_loss = 0.0
    for i, (videos, labels) in enumerate(train_loader):
        videos, labels = videos.to(DEVICE), labels.to(DEVICE)

        # --- SLOWFAST PATHWAY SPLIT ---
        # Fast pathway takes all 32 frames
        fast_pathway = videos
        # Slow pathway takes every 'ALPHA' frame (dim 2 is Time)
        slow_pathway = torch.index_select(
            videos,
            2,
            torch.linspace(0, videos.shape[2] - 1, videos.shape[2] // ALPHA).long().to(DEVICE)
        )

        # SlowFast expects a list of [slow_tensor, fast_tensor]
        inputs = [slow_pathway, fast_pathway]
        # ------------------------------

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}] Average Loss: {running_loss/len(train_loader):.4f}")

torch.save(model.state_dict(), "video_slowfast_model.pth")
print("Training Complete. Model Saved.")

# --- 6. TARGET FILE / INFERENCE ---
NEW_VIDEO_PATH = "/content/drive/MyDrive/video_data/test/v_PlayingCello_g04_c02.avi"

def run_guaranteed_inference(video_path):
    model.eval()

    try:
        # Load video
        vframes, _, _ = read_video(video_path, pts_unit='sec', output_format='TCHW')

        # Sample 32 frames
        t = vframes.shape[0]
        indices = torch.linspace(0, t - 1, NUM_FRAMES).long()
        vframes = vframes[indices]

        # Apply transformations
        vframes = slowfast_transform(vframes)

        # Permute: (T, C, H, W) -> (C, T, H, W)
        vframes = vframes.permute(1, 0, 2, 3)

        # Add Batch Dimension and move to GPU -> (1, C, T, H, W)
        input_tensor = vframes.unsqueeze(0).to(DEVICE)

        # Create pathways for inference
        fast_pathway = input_tensor
        slow_pathway = torch.index_select(
            input_tensor,
            2,
            torch.linspace(0, input_tensor.shape[2] - 1, input_tensor.shape[2] // ALPHA).long().to(DEVICE)
        )
        inference_inputs = [slow_pathway, fast_pathway]

        # Inference
        with torch.no_grad():
            output = model(inference_inputs)
            probs = F.softmax(output, dim=1)
            conf, pred_idx = torch.max(probs, 1)

        print("-" * 30)
        print(f"FILE: {os.path.basename(video_path)}")
        print(f"PREDICTED: {CLASSES[pred_idx.item()].upper()}")
        print(f"CONFIDENCE: {conf.item()*100:.2f}%")
        print("-" * 30)

    except Exception as e:
        print(f"Inference failed: {e}")

# Run it
run_guaranteed_inference(NEW_VIDEO_PATH)

Mounted at /content/drive
Downloading: "https://github.com/facebookresearch/pytorchvideo/zipball/main" to /root/.cache/torch/hub/main.zip
Downloading: "https://dl.fbaipublicfiles.com/pytorchvideo/model_zoo/kinetics/SLOWFAST_8x8_R50.pyth" to /root/.cache/torch/hub/checkpoints/SLOWFAST_8x8_R50.pyth


100%|██████████| 264M/264M [00:01<00:00, 248MB/s]



--- Starting Training (10 videos) ---


/usr/local/lib/python3.12/dist-packages/torchvision/io/_video_deprecation_warning.py:9: UserWarning: The video decoding and encoding capabilities of torchvision are deprecated from version 0.22 and will be removed in version 0.24. We recommend that you migrate to TorchCodec, where we'll consolidate the future decoding/encoding capabilities of PyTorch: https://github.com/pytorch/torchcodec
  warnings.warn(


Epoch [1/3] Average Loss: 0.6482
Epoch [2/3] Average Loss: 0.2709
Epoch [3/3] Average Loss: 0.1891
Training Complete. Model Saved.
------------------------------
FILE: v_PlayingCello_g04_c02.avi
PREDICTED: RUNNING
CONFIDENCE: 90.40%
------------------------------
